In [ ]:
# 1. Install Unsloth first
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install dependencies.
# We removed the specific 'xformers<0.0.27' constraint that was causing the build error.
# We also install xformers separately to ensure it grabs the correct binary wheel.
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install xformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-78r076ly/unsloth_87154870cf1740fb8464a4bd2e61e575
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-78r076ly/unsloth_87154870cf1740fb8464a4bd2e61e575
  Resolved https://github.com/unslothai/unsloth.git to commit 33b0343ec56595d4e7d7cdd25f173207ebf991b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 82.8 MB/s eta 0:00:00
 

In [ ]:
# Fix the trl version mismatch required by Unsloth
!pip install "trl>=0.18.2,<=0.24.0"

  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6


In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# 1. CONFIGURATION
max_seq_length = 2048
dtype = None
load_in_4bit = True

# 2. LOAD MODEL
print("Loading Phi-3.5 model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. ADD LoRA ADAPTERS
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. PREPARE YOUR DATA
# In Colab, uploaded files sit in the default folder or '/content/'
input_file = "clean_training_data.txt"

# Verify file exists before crashing
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Could not find {input_file}. Did you upload it to the Files sidebar?")

def prepare_dataset(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Split into chunks
    chunk_size = 2000
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    return Dataset.from_dict({"text": chunks})

dataset = prepare_dataset(input_file)
print(f"Dataset prepared with {len(dataset)} chunks.")

# 5. TRAIN
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",

    ),
)

print("Starting training...")
trainer_stats = trainer.train()

# 6. INFERENCE TEST
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [
        "User: Explain the concept of Pipelining in computer architecture.\nAssistant: "
    ], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Phi-3.5 model...
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,2.754700
2,2.662500
3,2.547700
4,2.744200
5,2.814400
6,2.670300
7,2.623900
8,2.667300
9,2.570500
10,2.479200


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇███
train/grad_norm,▁▁▁▁▁▁▂█▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▂▃▃▃▃▃▃▄▃▃▃▄▄▃▃▃
train/learning_rate,▁▂▄▇███▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁
train/loss,▇██▇▇▆▆▇▅▅▆▄▅▅▄▄▅▄▄▃▃▂▃▂▄▂▂▂▂▂▃▂▂▁▁▁▂▂▁▁
total_flos,5824348479590400.0
train/epoch,7.5
train/global_step,60
train/grad_norm,0.39433
train/learning_rate,0.0
train/loss,1.8666


['User: Explain the concept of Pipelining in computer architecture.\nAssistant:  Pipelining is a technique used in computer architecture to improve the performance of a processor. It is based on the concept of an assembly line, where each worker performs a small part of the overall task, and the output of one worker is the input of the next. In the case of a computer, the workers are functional units, and the task is to execute a sequence of instructions. Pipelining allows multiple instructions to be executed simultaneously by overlapping the execution of instructions. For example, if the processor takes 2 clock cycles to execute an instruction, then the processor can execute two instructions in every clock cycle, for']


In [ ]:
import torch # Ensure torch is imported

# 1. Fix the missing pad token issue globally
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

FastLanguageModel.for_inference(model)

print("Bot is ready! Type 'exit' to stop.")
print("-" * 30)

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    messages = [
        {"role": "user", "content": user_input}
    ]

    # Generate the input IDs
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    # Create an attention mask (1 for real data, 0 for padding)
    # Since we are doing 1-by-1 chat, everything is real data, so we use ones_like
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,  # <--- This fixes the warning
        pad_token_id=tokenizer.pad_token_id, # <--- Explicitly sets the pad token
        max_new_tokens=256, # we need to change this to make the response bigger and shorter for the time being , and we should think of something dynamic
        use_cache=True,
        temperature=0.3,
    )

    decoded_output = tokenizer.batch_decode(outputs)[0]

    # Clean up output
    try:
        response = decoded_output.split("<|assistant|>")[1].replace("<|end|>", "")
    except IndexError:
        response = decoded_output

    print(f"Bot: {response.strip()}")
    print("-" * 30)

Bot is ready! Type 'exit' to stop.
------------------------------
You: what is Computer Architecture 
Bot: Computer architecture refers to the design and structure of a computer system. It encompasses the hardware and the interface between the hardware and the software. Computer architecture is concerned with the design of the components of a computer system, the interconnections between those components, and the description of the interface between the hardware and the software.

Computer architecture can be divided into several sub-areas:

1. Instruction set architecture (ISA): the interface between the hardware and the high-level language programs. It defines the instruction set, the data types, the addressing modes, and the memory architecture.

2. Microarchitecture (or computer organization): the implementation of the ISA. It defines the datapath, the control unit, and the memory hierarchy.

3. Pipelining: the technique of overlapping the execution of instructions to increase thro

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save to your Google Drive folder
save_path = "/content/drive/My Drive/Colab_Models/Phi-3.5-COA-Finetune"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")

Mounted at /content/drive
Model saved to: /content/drive/My Drive/Colab_Models/Phi-3.5-COA-Finetune
